# 🔄 Module 04: Async Iterators, Generators & Context Managers

In production backend & data engineering, you rarely load millions of records into memory at once.
Instead, you stream data chunk-by-chunk using **Async Iterators** and **Async Generators**.

### What We Will Master:
1. **Writing Async Context Managers from Scratch**: `__aenter__` & `__aexit__`.
2. **Decorated Context Managers**: `@contextlib.asynccontextmanager`.
3. **Writing Async Iterators from Scratch**: `__aiter__` & `__anext__`.
4. **Async Generators** (`async def` with `yield`): streaming logs, paginated APIs, chunked S3 feeds.
5. **Async Comprehensions**: `[x async for x in gen()]`.
6. **🎯 Enterprise Challenge**: High-volume 100k-row chunked database stream without memory spikes.

---
## 1. Async Context Managers from Scratch (`__aenter__` / `__aexit__`)

Understanding the dunder protocol under the hood of `async with`.
- `__aenter__`: Must return a coroutine that yields the resource.
- `__aexit__(exc_type, exc_val, exc_tb)`: Must handle cleanup and return `True` if suppressing exceptions.

In [1]:
import asyncio
import time

class ManagedAsyncDbSession:
    def __init__(self, session_id: str):
        self.session_id = session_id

    async def __aenter__(self):
        print(f"[{time.strftime('%X')}] 🔌 Connected to Postgres session {self.session_id}")
        await asyncio.sleep(0.05) # Simulates network handshake
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print(f"[{time.strftime('%X')}] 🧹 Released Postgres session {self.session_id} back to connection pool cleanly!")
        await asyncio.sleep(0.02) # Simulates closing/returning connection
        if exc_type:
            print(f"⚠️ Error occurred during session: {exc_val}. Rolling back transaction!")
            return False # Re-raise exception

async with ManagedAsyncDbSession("0x10a2") as db:
    print("Executing query: SELECT * FROM transactions WHERE user_id=42")

[08:25:01] 🔌 Connected to Postgres session 0x10a2
Executing query: SELECT * FROM transactions WHERE user_id=42
[08:25:01] 🧹 Released Postgres session 0x10a2 back to connection pool cleanly!


---
## 2. Async Iterators from Scratch (`__aiter__` / `__anext__`)

An async iterator yields items asynchronously using `StopAsyncIteration` to signal the end of the stream.

In [2]:
class AsyncChunkReader:
    def __init__(self, total_chunks: int = 3, chunk_size: int = 5):
        self.total_chunks = total_chunks
        self.chunk_size = chunk_size
        self.current = 0

    def __aiter__(self):
        return self

    async def __anext__(self):
        if self.current >= self.total_chunks:
            raise StopAsyncIteration
        self.current += 1
        await asyncio.sleep(0.1) # Simulates streaming next network buffer
        start_row = (self.current - 1) * self.chunk_size + 1
        end_row = self.current * self.chunk_size
        return f"Received rows {start_row} to {end_row}"

async for chunk in AsyncChunkReader(total_chunks=3):
    print(f"Chunk: {chunk}")
print("✅ Stream finished!")

Chunk 1: Received rows 1 to 5
Chunk 2: Received rows 6 to 10
Chunk 3: Received rows 11 to 15
✅ Stream finished!


---
## 3. Async Generators & Async Comprehensions

Async generators combine `async def` with `yield`.
They are the cleanest, most idiomatic way in modern Python to write data pipelines.

In [3]:
async def trade_ticker_stream():
    trades = [
        {"trade_id": 101, "amount": 15000},
        {"trade_id": 102, "amount": 4500},
        {"trade_id": 103, "amount": 22000},
        {"trade_id": 104, "amount": 1200},
    ]
    for t in trades:
        await asyncio.sleep(0.05) # Simulates incoming WebSocket tick
        yield t

# Async List Comprehension:
high_value_trades = [
    trade async for trade in trade_ticker_stream()
    if trade["amount"] >= 10000
]
print("Filtered High Value Trades:", high_value_trades)

Filtered High Value Trades: [{'trade_id': 101, 'amount': 15000}, {'trade_id': 103, 'amount': 22000}]
